In [39]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

# 定义模型
这里要识别图片，所以需要定义多模态模型，不能使用DeepSeek


In [40]:
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

load_dotenv()

model = ChatOpenAI(
    model="qwen3.5-plus",
    openai_api_key=os.getenv("DASHSCOPE_API_KEY"),
    openai_api_base=os.getenv("DASHSCOPE_BASE_URL"),
    temperature=0.2
)


# 定义工具


In [41]:
from langchain_tavily import TavilySearch

# web搜索工具，使用tavily作为web搜索工具
web_search = TavilySearch(
    max_results=5,
    topic="general",
)

# 添加记忆管理

这里我们采用sqllite

In [42]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# 连接sqlite
connection = sqlite3.connect("resources/personal_chief.db", check_same_thread=False)
# 初始化checkpointer
checkpointer = SqliteSaver(connection)
# 自动建表
checkpointer.setup()

# 定义智能体

In [43]:
from langchain.agents import create_agent

system_prompt = """
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：
1.识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用食材清单”。
2.智能食谱检索：优先调用 web_search 工具，以“可用食材清单”为核心关键词，查找可行菜谱。
3.多维度评估与排序：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4.结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由、食谱的参考图片，帮助用户快速做出决策。

请严格按照流程，优先调用 web_search 工具搜索食谱，搜索不到的情况下才能自己发挥。
"""

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=checkpointer
)

# 测试

我们直接使用在线图片来测试，图片如下：

<img width="320" src="https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg">


In [44]:
from langchain.messages import HumanMessage

multimodal_message = HumanMessage([
        {"type": "text", "text": "帮我看看能做什么."},
        {"type": "image", "url": "https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg"},
    ])

config = {"configurable": {"thread_id": "6"}}

In [45]:
response = agent.invoke({"messages": [multimodal_message]}, config)

In [24]:
# 友好打印
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么.'}, {'type': 'image', 'url': 'https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg'}]
================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么.'}, {'type': 'image', 'url': 'https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg'}]
================================== Ai Message ==================================

我先把冰箱里的食材仔细看一遍，再去网上找找合适的菜谱。

**🔍 第一步：识别食材（冰箱实况）**

我先把整层看一遍，同时开始检索菜谱：
Tool Calls:
  tavily_search (call_00_itmWQUFyQuAHwme5c6PP9635)
 Call ID: call_00_itmWQUFyQuAHwme5c6PP9635
  Args:
    query: 西红柿 茄子 青椒 豆角 家常菜谱 做法
    search_depth: advanced
    include_images: True
  tavily_search (call_01_3J96qQgHIOQrHBt9pWcO0229)
 Call ID: call_01_3J96qQgHIOQrHBt9pWcO0229
  Args:
    query: 生菜 白菜 蒜蓉 清炒 简单家常菜谱
    search_depth: advanced
    include_images: True
  tavily

In [25]:
response = agent.invoke(
    {"messages": [HumanMessage(content="我喜欢第3道菜，可以说的更详细点吗？")]},
    config
)

In [10]:
# 友好打印
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么.'}, {'type': 'image', 'url': 'https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg'}]
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_62b304aaa0c846ae9d943b04)
 Call ID: call_62b304aaa0c846ae9d943b04
  Args:
    query: 豆腐 西红柿 青椒 西兰花 茄子 虾 腊肠 家常菜谱 简单营养
    search_depth: advanced
    topic: general
================================= Tool Message =================================
Name: tavily_search

{"query": "豆腐 西红柿 青椒 西兰花 茄子 虾 腊肠 家常菜谱 简单营养", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.youtube.com/watch?v=HGVg93qXRBA", "title": "豆腐，西红柿，鸡蛋一起上锅蒸一蒸，教你没吃过的做法 - YouTube", "content": "豆腐，西红柿，鸡蛋一起上锅蒸一蒸，教你没吃过的做法，好吃不上火，用料最简单，营养全保留，太鲜了#like #美食教程 #food #cooking. 10K views · 5", "score": 0.9986436, "raw_content": null}, {"url": "https://www.youtub

In [26]:
# 友好打印
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

## 🍆 第3道菜完整详解：茄子炒西红柿（加青椒）

这道菜的评分是 85 分，在营养方面排第一。它之所以排名不最高，只是因为有两个容易做坏的地方：**茄子吸油、茄子出水变烂**。下面我会讲清楚每一步该怎么做。

---

### 一、食材准备（2 人份）

| 食材 | 用量 | 处理要求 |
|---|---|---|
| 长紫茄 | 1 根（约 300g） | 就是你冰箱底层那根，选**表皮光亮、捏着有弹性的**，皮发暗、一按就塌的别用 |
| 西红柿 | 2 个（约 350g） | 从 4 个里挑**最红最软**的两个，熟番茄才出得了汁 |
| 螺丝椒、青椒 | 1 根、半个 | 螺丝椒负责辣香，青椒负责脆感，可以只放一种 |
| 蒜 | 4 瓣 | 拍碎再切末，**现切现用**，切早了会发苦 |
| 小葱 | 1 根 | 葱白切段爆香用，葱绿最后撒 |
| 香菜 | 可选 1 小把 | 出锅撒，不爱吃可以不放 |

**调味料（建议提前倒进小碗调匀）**

- 盐 3g（约半平勺）
- 鸡精 2g
- 生抽 15ml（1 汤勺）
- 老抽 2ml（几滴，只用于上色，多了会发黑）
- 蚝油 5ml（1 小勺）
- 香油 5ml
- 额外备：**腌茄子用的盐 8g**，这个单独放，不要和调味盐混在一起

---

### 二、关键前置：茄子的“杀水”处理（这步做对了，就成功了 70%）

1. **去皮**：长紫茄皮比较薄，也可以保留，但**保留皮会更难炒软，也更容易发黑**。第一次做建议削掉皮，成功率更高。
2. **切条**：切掉两头，先切成 **0.8cm 厚的片**，再切成 **0.8cm 宽、6cm 长的条**。粗细要均匀，不然细的先烂、粗的还生。
3. **加盐抓匀**：倒进大盆，撒 8g 盐，用手抓拌 30 秒，让每根都沾上盐。
4. **静置 10 分钟**：期间茄子会出很多水，明显变软，体积缩小。这段时间正好去切西红柿和青椒。
5. **挤水、冲洗、再挤干**：
   - 用手**轻轻**挤掉多余水分，不要用死力，也别把茄子捏碎
   - 放进清水里过一遍，洗掉多余的盐分，不然成菜

# 流式调用

有工具调用的agent消息类型比较多，要正确的流式输出比较麻烦，这里给出一个示例


In [46]:
from langchain.messages import AIMessageChunk

for chunk, metadata in agent.stream(
            {"messages": [multimodal_message]},
            config,
            stream_mode="messages"
        ):
            if isinstance(chunk, AIMessageChunk) and chunk.content:
                print(chunk.content, end="", flush=True)

## 🧊 冰箱食材快速总结

看到你再次发照片，我确认一下：**之前已经详细讲解过「茄子炒西红柿」的完整做法了**（包括茄子杀水、不吸油的技巧等）。

这次我快速帮你梳理**还能做什么其他菜**：

---

### 🎯 冰箱核心食材（按优先级）

| 类别 | 食材 | 建议 |
|---|---|---|
| **蛋白质** | 豆腐1块、虾仁1盒 | 2天内用完，优先做 |
| **叶菜类** | 生菜、香菜、白菜 | 1-2天内用完，易黄 |
| **茄果类** | 西红柿4个、茄子1根、青椒3个 | 3-5天内 |
| **根茎类** | 紫薯、玉米、西兰花 | 较耐放，5-7天 |
| **调味** | 腊肠笋子酱、辣椒酱 | 常备 |

---

###  除了茄子炒西红柿，还能做这些

| 菜谱 | 用时 | 难度 | 核心食材 |
|---|---|---|---|
| 虾仁豆腐蒸蛋 | 15分钟 | ⭐ | 虾仁+豆腐+鸡蛋 |
| 西兰花虾仁豆腐 | 20分钟 | ⭐⭐ | 西兰花+虾仁+豆腐 |
| 蒜蓉炒生菜 | 5分钟 | ⭐ | 生菜+蒜 |
| 腊肠笋子酱拌饭 | 5分钟 | ⭐ | 酱+米饭+白灼菜 |
| 干煸豆角 | 15分钟 | ⭐⭐⭐ | 豆角 |

---

### ❓ 你想做什么？

请告诉我：

1. **想继续了解茄子炒西红柿** → 我之前的详细步骤还在，有任何问题可以问我
2. **想试试其他菜** → 告诉我你对哪道感兴趣，我详细讲
3. **想让我帮你搭配一整套菜单** → 告诉我你有几人吃、有没有米饭/面条等主食

这样我能更精准地帮你～ 😊

# 获取会话历史

既然会话记忆交给了checkpointer管理，我们自然可以从中获取会话历史


In [47]:
checkpointer.get(config)

{'v': 4,
 'ts': '2026-11-10T10:05:47.521941+00:00',
 'id': '1f1dcee4-cdf3-65e0-801c-7d47a0a7f0af',
 'channel_values': {'messages': [HumanMessage(content=[{'type': 'text', 'text': '帮我看看能做什么.'}, {'type': 'image', 'url': 'https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg'}], additional_kwargs={}, response_metadata={}, id='35c18446-ca8a-446a-8b32-d9ceecd9bc3c'),
   HumanMessage(content=[{'type': 'text', 'text': '帮我看看能做什么.'}, {'type': 'image', 'url': 'https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg'}], additional_kwargs={}, response_metadata={}, id='32659613-5335-4491-8fde-c87368c074c8'),
   AIMessage(content='我先把冰箱里的食材仔细看一遍，再去网上找找合适的菜谱。\n\n**🔍 第一步：识别食材（冰箱实况）**\n\n我先把整层看一遍，同时开始检索菜谱：', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1579, 'prompt_tokens': 4039, 'total_tokens': 5618, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 1

In [48]:
for m in checkpointer.get(config)['channel_values']['messages']:
    print(type(m))
    print(m)

<class 'langchain_core.messages.human.HumanMessage'>
content=[{'type': 'text', 'text': '帮我看看能做什么.'}, {'type': 'image', 'url': 'https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg'}] additional_kwargs={} response_metadata={} id='35c18446-ca8a-446a-8b32-d9ceecd9bc3c'
<class 'langchain_core.messages.human.HumanMessage'>
content=[{'type': 'text', 'text': '帮我看看能做什么.'}, {'type': 'image', 'url': 'https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg'}] additional_kwargs={} response_metadata={} id='32659613-5335-4491-8fde-c87368c074c8'
<class 'langchain_core.messages.ai.AIMessage'>
content='我先把冰箱里的食材仔细看一遍，再去网上找找合适的菜谱。\n\n**🔍 第一步：识别食材（冰箱实况）**\n\n我先把整层看一遍，同时开始检索菜谱：' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 1579, 'prompt_tokens': 4039, 'total_tokens': 5618, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 1215, 'rejected_prediction_tokens'

定义一个便捷的获取会话历史的函数：


In [49]:
from langchain.messages import AIMessage

def get_messages(thread_id: str) -> list[dict[str, str]]:
    """获取会话历史"""

    # 根据 thread_id 查询 checkpoint
    cp = checkpointer.get({"configurable": {"thread_id": thread_id}})

    # 如果不存在，返回空列表
    if not cp:
        return []

    # 安全获取 messages
    channel_values = cp.get("channel_values")
    if not channel_values:
        return []

    messages = channel_values.get("messages", [])
    if not messages:
        return []

    # 转换消息格式
    result = []
    for msg in messages:
        if not msg.content:
            continue

        if isinstance(msg, HumanMessage):
            result.append({"role": "user", "content": msg.content})
        elif isinstance(msg, AIMessage):
            result.append({"role": "assistant", "content": msg.content})

    return result

In [50]:
get_messages("6")

[{'role': 'user',
  'content': [{'type': 'text', 'text': '帮我看看能做什么.'},
   {'type': 'image',
    'url': 'https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg'}]},
 {'role': 'user',
  'content': [{'type': 'text', 'text': '帮我看看能做什么.'},
   {'type': 'image',
    'url': 'https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg'}]},
 {'role': 'assistant',
  'content': '我先把冰箱里的食材仔细看一遍，再去网上找找合适的菜谱。\n\n**🔍 第一步：识别食材（冰箱实况）**\n\n我先把整层看一遍，同时开始检索菜谱：'},
 {'role': 'assistant',
  'content': '## 🧊 一、当前可用食材清单\n\n我把冰箱上层到保鲜抽屉逐层看了一遍，整理如下：\n\n**蔬菜类**\n| 食材 | 位置 | 数量与状态 |\n|---|---|---|\n| 西红柿 | 第二层 | 4个，红透饱满，状态很好 |\n| 长条绿辣椒（螺丝椒）+ 青椒 | 第二层 | 2根+1个，表皮紧实鲜亮 |\n| 豆角或豇豆 | 第二层右（袋装） | 约1把，部分略发蔫，建议2天内用完 |\n| 香菜 | 第二层右 | 1小把，尚新鲜，容易变黄、要早用 |\n| 生菜 | 第二层 | 1~2棵，叶片挺脆 |\n| 大白菜或包菜 | 底层抽屉 | 1大棵，外层叶略老，剥掉即可 |\n| 青皮南瓜（或冬瓜） | 第二层 | 1个，深绿皮、无软斑，耐放 |\n| 玉米（带皮） | 第二层 | 1根，建议尽快吃，放久会失水变硬 |\n| 茄子 | 底层抽屉 | 1根紫长茄，表皮光亮、有弹性 |\n| 小葱 | 底层 | 1把，葱白还硬挺，适合当配料 |\n| 紫薯或小红薯 | 第三层盒装 | 4~5个，耐放，可当主食 |\n| 浅色

# 清空会话历史


In [32]:
checkpointer.delete_thread("6")